**Choix du modèle :**
- Contrainte matériel RTX3050 8Gb
- Possibilité de faire du multivarié


**PatchTST:**
- leger en memeoire, bonne mémoire long terme
- Facile a entrainer notamment grace aux patching
- possibilite de multivariée avec covariables en channels (channel-independent = False si tu veux interactions entre variables)
- juste un encoder qui predit un horizon d'un coup
- 1 token n'est pas un pas de temps, mais un patch de temps (allégement)
- dépendant aux tailles de patchs qui doivent etre choisis intelligemment en fonction du contexte de la TS

**Informer (sparse) / AutoFormer (decompo trend/seasonality) /FEDformer (Fourier, fort sur signaux périodiques) :** meilleure gestion long terme

**TFT :** gere parfaitement les covariables, interpretable, mais lent et lourd en mémoire!

**iTransformer :**
- focus sur les relations entre variables (multivariée)
- d'après nos précédents modèles, les covariables ont pratiquement aucun effet sur la prediction de la target
- mais ici, ne mesure pas seulement comment chaque covariable evoluent dans le temps, mais apprend les relations entre les covariables par du cross attention, et non pour prédire le temps (les tokens sont les variables), complexité O(N²) avec N=nbr variables, en faible nombre pour notre dataset, ce qui en fait un modèle très léger.
- moins puissant en terme de mémoire temporelle
- dépendance a la qualité des covariables, et aux interactions entre les variables
- ne modélise pas l'incertitude


**Conclu :** pas besoin d'une trop longue mémoire car forte autocorrélation des valeurs, modele leger en mémoire PatchTST permet des plus grands batchs. PatchTST est davantage robuste car il ne fait pa s l'hypothese forte de la tendance+saison, donc meilleur sur données plus bruitées et moins dependant du tuning (VS FFT+attention etc...). Le FEDformer doit également appliquer son hypothese sur les covariables. FEDformer est trop exposé au biais inductif de la périodicité, et peut sous exploiter les covariables par extension. AutoFormer est plus souple que FED, et plus généraliste, plus indépendant de la structure initiale du signal. Attention, cependanjt output deterministe, pas de mesures d'incertitues (<> distribution).
Puisqu'il est difficile de juger la pertinence des covariables dans la prédiction de la target, on écarte iTransformer et on choisit PatchTST pour sa polyvalence (uni et mutlivariée). De plus, le papier de iTransformer montre qu'en moyenne sur le dataset ETT, le MSE et le MAE sont légèrement meilleur pour PatchTST (0.381 VS 0.383 et 0.397 VS 0.399).


**Sources :**
- https://www.geeksforgeeks.org/deep-learning/transformer-for-time-series-forecasting/
- https://medium.com/@serana.ai/transformers-for-time-series-forecasting-e5e0327e78be
- A Closer Look at Transformers for Time Series Forecasting: Understanding Why They Work and Where They Struggle (Yu Chen, Nathalia Cespedes, Payam Barnaghi)
- Liu et al. Itransformer: Inverted Transformers are Effective for Time Series Forecasting, (2024)
- Nie et al. A Time Series is Worth 64 Words: Long-Term Forecasting With Transformers, (2023)
- Wang, Y., Wu, H., Dong, J., Liu, Y., Long, M., and Wang, J. Deep time series models: A comprehensive survey and benchmark. arXiv preprint arXiv:2407.13278, 2024b.

# Model

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../"))  # ou le bon chemin vers utils
sys.path.append("../models")

from utils.exp.exp_long_term_forecasting import Exp_Main
import numpy as np
from argparse import Namespace
import torch
import random

seed=2021
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

############# MULTIVARIATE #############
# for pred_len in [24, 96, 192]
# features = MS : multivariate predict univariate OR S : univariate predict univariate
data="ETTh1"
pred_len=96

args = Namespace(
    seed=seed,
    pred_len=pred_len,
    data=data,
    root_path='../data/ETT/',
    data_path='ETTh1.csv',
    model_id = f"{data}_{pred_len}",
    model='PatchTST',
    features="MS",
    target="OT",
    freq='h',
    checkpoints="../../outputs/checkpoints/PatchTST",
    is_training=False,
    seq_len=336,
    label_len=48,

    # PatchTST
    fc_dropout=0.3,
    head_dropout=0,
    patch_len=16,
    stride=8,
    padding_patch='end',
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    # Formers
    enc_in=7,
    e_layers=3,
    n_heads=4,
    d_model=16,
    d_ff=128,
    dropout=0.3,
    embed_type=0,
    dec_in=7,
    c_out=7,
    d_layers=1,
    moving_avg=25,
    factor=1,
    distil=True,
    embed='timeF',
    activation='gelu',
    output_attention=False,
    do_predict=False,

    # optimization
    des='Exp',
    train_epochs=100,
    itr=1,
    batch_size=128,
    learning_rate=0.0001,
    num_workers=8,
    patience=100,
    loss='mse',
    lradj='type3',
    pct_start=0.3,
    use_amp=True,

    # GPU
    use_gpu=True,
    gpu=0,
    use_multi_gpu=False,
    devices='0,1,2,3',
    test_flop=False
)

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.dvices = args.devices.replace(' ', '')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

task_name = 'long_term_forecast'
print('Long Term Forecast')
Exp = Exp_Main
print('Exp:', Exp)


if args.is_training:
    for ii in range(args.itr):
        # setting record of experiments
        setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
                args.model_id,
                args.model,
                args.data,
                args.features,
                args.seq_len,
                args.label_len,
                args.pred_len,
                args.d_model,
                args.n_heads,
                args.e_layers,
                args.d_layers,
                args.d_ff,
                args.factor,
                args.embed,
                args.distil,
                args.des,
                ii)

        exp = Exp(args)  # set experiments
        print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
        exp.train(setting)

        print('>>>>>>>validating : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
        exp.test(setting)

        if args.do_predict:
            print('>>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
            exp.predict(setting, True)

        torch.cuda.empty_cache()
else:
    ii = 0
    setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(args.model_id,
                                                                                                    args.model,
                                                                                                    args.data,
                                                                                                    args.features,
                                                                                                    args.seq_len,
                                                                                                    args.label_len,
                                                                                                    args.pred_len,
                                                                                                    args.d_model,
                                                                                                    args.n_heads,
                                                                                                    args.e_layers,
                                                                                                    args.d_layers,
                                                                                                    args.d_ff,
                                                                                                    args.factor,
                                                                                                    args.embed,
                                                                                                    args.distil,
                                                                                                    args.des,
                                                                                                    ii)

    exp = Exp(args)  # set experiments
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting, test=1)
    torch.cuda.empty_cache()



# from official paper's github PatchTST : https://github.com/yuqinie98/PatchTST/tree/main
# model = PatchTST(configs, max_seq_len=336, d_k=None, d_v=None, norm='BatchNorm', attn_dropout=0.,
#                  act="gelu", key_padding_mask='auto', padding_var=None, attn_mask=None, res_attention=True,
#                  pre_norm=False, store_attn=False, pe='zeros', learn_pe=True, pretrain_head=False, head_type='flatten', verbose=False)

Long Term Forecast
Exp: <class 'utils.exp.exp_long_term_forecasting.Exp_Main'>
Use GPU: cuda:0
>>>>>>>testing : ETTh1_96_PatchTST_ETTh1_ftMS_sl336_ll48_pl96_dm16_nh4_el3_dl1_df128_fc1_ebtimeF_dtTrue_Exp_0<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
test 2785
loading model
mse:0.054660238325595856, mae:0.17838245630264282, rse:0.6838446259498596
